# GuitarSet Audio→MIDI Fine-Tuning V1

This notebook is a **guitar-specific audio-to-MIDI training experiment** built from the same path/split/parsing style as the working Basic Pitch notebooks.

Goal: improve `P(MIDI | Audio)` only. It does **not** do fretboard assignment/tab generation. The output is MIDI note events that can plug into the existing pipeline:

```text
audio → guitar-specific MIDI model → MIDI notes → existing Viterbi/fretboard assignment → ASCII tab/UI
```

What this tests:
- Train a lightweight guitar-specific model on GuitarSet labels
- CQT input features, similar to guitar-transcription papers
- Frame-level multi-pitch supervision over guitar MIDI range
- Validation-set threshold tuning
- Note-event extraction + pitch/onset F1 evaluation
- Optional feature augmentation / pseudo-label hooks

This is not a literal fine-tune of Basic Pitch's internal weights. The Basic Pitch pip package does not expose a simple fine-tuning API. This notebook builds a compatible **Basic Pitch replacement/student model** for the audio→MIDI module.


In [1]:
# Install dependencies when running in Colab.
# Runtime → Change runtime type → GPU is recommended.
import sys, subprocess, importlib.util

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, import_name in [
    ('librosa>=0.10.1', 'librosa'),
    ('soundfile', 'soundfile'),
    ('mido', 'mido'),
    ('torch', 'torch'),
    ('tqdm', 'tqdm'),
    ('scikit-learn', 'sklearn'),
]:
    if importlib.util.find_spec(import_name) is None:
        print('Installing', pkg)
        pip_install(pkg)


Installing mido


### Fixed CQT config

This version fixes the previous CQT crash by using `N_BINS = 144` from `FMIN = E2` at `SR = 22050`. The prior `192`-bin setting exceeded the Nyquist frequency. If you change CQT settings later, set `CLEAR_FEATURE_CACHE = True` for one run.

In [2]:
# Mount Google Drive when running in Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ModuleNotFoundError:
    print('Not running in Colab. Skipping Google Drive mount.')


Mounted at /content/drive
Google Drive mounted successfully.


In [3]:
from pathlib import Path
import os, json, math, random, warnings, time
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)


DEVICE: cuda


## 1. Config

Paths are intentionally copied from the working notebooks. If your Drive layout changes, only update the candidate lists here.


In [4]:
# -------------------------
# USER CONFIG
# -------------------------
CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'guitarset_audio_to_midi_finetune_v1_fixed_cqt'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
    Path('/content/drive/MyDrive/GuitarSet'),
    Path('/content/drive/MyDrive/Capstone'),
    Path('/content/drive/MyDrive'),
    Path('/mnt/data/fretwork_repo/GuitarSet'),
    Path('/mnt/data/fretwork_repo'),
]

AUDIO_ROOT_CANDIDATES = [
    CAPSTONE_ROOT / 'GuitarSet' / 'Audio',
    CAPSTONE_ROOT / 'GuitarSet' / 'AudioFiles',
    CAPSTONE_ROOT / 'FullGuitarSetData' / 'AudioFiles',
    CAPSTONE_ROOT / 'FullGuitarSetData' / 'Audio',
    CAPSTONE_ROOT / 'Audio',
    CAPSTONE_ROOT / 'GuitarSet',
    CAPSTONE_ROOT,
    Path('/content/drive/MyDrive/GuitarSet/Audio'),
    Path('/content/drive/MyDrive/GuitarSet'),
]

AUDIO_EXTENSIONS = ['.wav', '.mp3', '.m4a', '.flac', '.ogg']

# Audio/features
SR = 22050
HOP_LENGTH = 512
BINS_PER_OCTAVE = 24
N_OCTAVES = 6
N_BINS = BINS_PER_OCTAVE * N_OCTAVES  # 144 bins; safe under Nyquist at sr=22050
FMIN = librosa.note_to_hz('E2')  # guitar low E (~82 Hz)

# Sanity-check CQT top frequency. 192 bins from E2 goes above Nyquist at 22.05 kHz,
# so we use 144 bins by default. This covers guitar fundamentals + useful harmonics.
FMAX_APPROX = FMIN * (2 ** (N_BINS / BINS_PER_OCTAVE))
NYQUIST = SR / 2
assert FMAX_APPROX < NYQUIST, f'CQT max freq {FMAX_APPROX:.1f} exceeds Nyquist {NYQUIST:.1f}; reduce N_BINS or FMIN.'

# MIDI output range. 40-88 matches prior Basic Pitch eval.
MIN_MIDI = 40
MAX_MIDI = 88
N_PITCHES = MAX_MIDI - MIN_MIDI + 1

# Split, same style as prior notebooks.
SPLIT_SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# Debug controls. Set these to small numbers for a smoke test.
MAX_TRAIN_RECORDINGS = None
MAX_VAL_RECORDINGS = None
MAX_TEST_RECORDINGS = None

# Training controls. Start small; increase EPOCHS once it runs end-to-end.
WINDOW_SECONDS = 8.0
WINDOW_HOP_SECONDS = 4.0
BATCH_SIZE = 12
EPOCHS = 20
LR = 3e-4
NUM_WORKERS = 0

# Augmentation controls.
USE_FEATURE_AUGMENTATION = True
AUG_PROB = 0.50
PITCH_SHIFT_AUG_SEMITONES = [-2, -1, 0, 1, 2]  # feature-roll approximation; only train set

# Cache controls. Set True if changing CQT settings or labels.
CLEAR_FEATURE_CACHE = False

# Eval controls.
ONSET_TOLERANCE_SECONDS = 0.05
OFFSET_MIN_DURATION_SECONDS = 0.03
THRESHOLD_GRID = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]
MIN_DURATION_GRID = [0.02, 0.03, 0.05, 0.08]

CACHE_DIR = OUTPUT_DIR / 'cache'
FEATURE_CACHE_DIR = CACHE_DIR / 'cqt_features'
MODEL_DIR = OUTPUT_DIR / 'models'
for d in [CACHE_DIR, FEATURE_CACHE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('OUTPUT_DIR:', OUTPUT_DIR)


OUTPUT_DIR: /content/drive/MyDrive/Capstone/outputs/guitarset_audio_to_midi_finetune_v1_fixed_cqt


## 2. GuitarSet JAMS parsing and split

This mirrors the working notebooks and reads `.jams` directly, without the external `jams` package.


In [5]:
def find_jams_dir(data_root):
    candidates = [
        data_root / 'JamsFiles',
        data_root / 'Annotations',
        data_root / 'GuitarSet' / 'Annotations',
        data_root / 'FullGuitarSetData' / 'JamsFiles',
        data_root / 'FullGuitarSetData' / 'Annotations',
        data_root,
    ]
    for c in candidates:
        if c.exists() and list(c.glob('*.jams')):
            return c
    if data_root.exists():
        try:
            for match in data_root.rglob('*.jams'):
                return match.parent
        except Exception as e:
            print(f'Could not recursively search {data_root}: {e}')
    return None


def choose_data_root_and_jams_dir(candidates):
    checked = []
    for root in candidates:
        checked.append((root, root.exists()))
        if not root.exists():
            continue
        jams_dir = find_jams_dir(root)
        if jams_dir is not None:
            return root, jams_dir
    print('Could not find .jams files automatically.')
    for root, exists in checked:
        print(f' - {root} | exists: {exists}')
    raise FileNotFoundError('Could not find .jams files. Update DATA_ROOT_CANDIDATES.')


def get_annotation_data(annotation):
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []


def parse_string_from_data_source(data_source):
    try:
        s = int(data_source)
        return s if 0 <= s <= 5 else None
    except Exception:
        return None


def parse_guitarset_jams(jams_path):
    with open(jams_path, 'r') as f:
        jam = json.load(f)
    recording = jams_path.stem
    notes = []
    annotations = jam.get('annotations', [])
    for ann in annotations:
        namespace = ann.get('namespace', '')
        if namespace != 'note_midi':
            continue
        data_source = ann.get('annotation_metadata', {}).get('data_source')
        string_idx = parse_string_from_data_source(data_source)
        for item in get_annotation_data(ann):
            try:
                onset = float(item.get('time', 0.0))
                duration = float(item.get('duration', 0.0))
                pitch = int(round(float(item.get('value'))))
            except Exception:
                continue
            if duration <= 0:
                continue
            if MIN_MIDI <= pitch <= MAX_MIDI:
                notes.append({
                    'recording': recording,
                    'onset': onset,
                    'offset': onset + duration,
                    'duration': duration,
                    'pitch': pitch,
                    'string': string_idx,
                })
    notes = sorted(notes, key=lambda x: (x['onset'], x['pitch']))
    return {'recording': recording, 'jams_path': str(jams_path), 'notes': notes}

DATA_ROOT, JAMS_DIR = choose_data_root_and_jams_dir(DATA_ROOT_CANDIDATES)
JAMS_FILES = sorted(JAMS_DIR.glob('*.jams'))
print(f'DATA_ROOT selected: {DATA_ROOT}')
print(f'Found {len(JAMS_FILES)} JAMS files in {JAMS_DIR}')

records = [parse_guitarset_jams(p) for p in tqdm(JAMS_FILES, desc='Parsing JAMS')]
records = [r for r in records if len(r['notes']) > 0]
print('Parsed records with notes:', len(records))
print('Example:', records[0]['recording'], 'notes:', len(records[0]['notes']))


DATA_ROOT selected: /content/drive/MyDrive/Capstone/FullGuitarSetData
Found 360 JAMS files in /content/drive/MyDrive/Capstone/FullGuitarSetData/JamsFiles


Parsing JAMS:   0%|          | 0/360 [00:00<?, ?it/s]

Parsed records with notes: 360
Example: 00_BN1-129-Eb_comp notes: 133


In [6]:
def split_records_by_recording(records, train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    if not np.isclose(train_frac + val_frac + test_frac, 1.0):
        raise ValueError('Fractions must sum to 1.0')
    rng = random.Random(seed)
    groups = {
        'solo': [r for r in records if r['recording'].endswith('_solo')],
        'comp': [r for r in records if r['recording'].endswith('_comp')],
        'other': [r for r in records if not (r['recording'].endswith('_solo') or r['recording'].endswith('_comp'))],
    }
    train, val, test = [], [], []
    for label, group in groups.items():
        group = list(group)
        rng.shuffle(group)
        n = len(group)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        train.extend(group[:n_train])
        val.extend(group[n_train:n_train+n_val])
        test.extend(group[n_train+n_val:])
    return train, val, test

TRAIN_RECORDS, VAL_RECORDS, TEST_RECORDS = split_records_by_recording(
    records, TRAIN_FRAC, VAL_FRAC, TEST_FRAC, seed=SPLIT_SEED
)

if MAX_TRAIN_RECORDINGS is not None:
    TRAIN_RECORDS = TRAIN_RECORDS[:MAX_TRAIN_RECORDINGS]
if MAX_VAL_RECORDINGS is not None:
    VAL_RECORDS = VAL_RECORDS[:MAX_VAL_RECORDINGS]
if MAX_TEST_RECORDINGS is not None:
    TEST_RECORDS = TEST_RECORDS[:MAX_TEST_RECORDINGS]

print('Split sizes:')
print('TRAIN:', len(TRAIN_RECORDS))
print('VAL:', len(VAL_RECORDS))
print('TEST:', len(TEST_RECORDS))


Split sizes:
TRAIN: 252
VAL: 54
TEST: 54


## 3. Pair audio with records

Uses the same stem matching as the Basic Pitch eval notebooks.


In [7]:
def find_audio_files(audio_roots, exts=AUDIO_EXTENSIONS):
    audio_by_stem = {}
    for root in audio_roots:
        root = Path(root)
        if not root.exists():
            continue
        for ext in exts:
            try:
                for p in root.rglob(f'*{ext}'):
                    audio_by_stem.setdefault(p.stem, p)
            except Exception as e:
                print(f'Could not search {root}: {e}')
    return audio_by_stem


def audio_candidates_for_recording(recording):
    recording = str(recording)
    cands = [
        recording,
        f'{recording}_mic',
        f'{recording}_hex',
        recording.replace('_solo', '_solo_mic'),
        recording.replace('_comp', '_comp_mic'),
        recording.replace('_solo', '_solo_hex'),
        recording.replace('_comp', '_comp_hex'),
    ]
    seen, out = set(), []
    for c in cands:
        if c not in seen:
            out.append(c); seen.add(c)
    return out


def find_audio_for_record(record):
    for stem in audio_candidates_for_recording(record['recording']):
        if stem in AUDIO_BY_STEM:
            return AUDIO_BY_STEM[stem]
    return None

AUDIO_BY_STEM = find_audio_files(AUDIO_ROOT_CANDIDATES)
print(f'Found {len(AUDIO_BY_STEM)} audio files.')

for split_name, split_records in [('train', TRAIN_RECORDS), ('val', VAL_RECORDS), ('test', TEST_RECORDS)]:
    paired = []
    missing = []
    for r in split_records:
        p = find_audio_for_record(r)
        if p is None:
            missing.append(r['recording'])
        else:
            paired.append((r, p))
    globals()[f'{split_name.upper()}_PAIRED'] = paired
    print(f'{split_name}: paired {len(paired)} / {len(split_records)}')
    if missing[:5]:
        print('  missing examples:', missing[:5])


Found 636 audio files.
train: paired 252 / 252
val: paired 54 / 54
test: paired 54 / 54


## 4. Feature and label cache

Each record is converted to:
- `X`: log-CQT feature matrix with shape `(freq_bins, frames)`
- `Y`: pitch-roll labels with shape `(pitches, frames)`, for MIDI `40..88`

This is the supervised GuitarSet version of `P(MIDI | Audio)`.


In [8]:
def record_cache_path(recording):
    return FEATURE_CACHE_DIR / f'{recording}_cqt_pitchroll.npz'


def make_pitch_roll(notes, n_frames, sr=SR, hop_length=HOP_LENGTH):
    Y = np.zeros((N_PITCHES, n_frames), dtype=np.float32)
    for note in notes:
        pitch = int(note['pitch'])
        if pitch < MIN_MIDI or pitch > MAX_MIDI:
            continue
        start = max(0, int(np.floor(note['onset'] * sr / hop_length)))
        end = min(n_frames, int(np.ceil(note['offset'] * sr / hop_length)))
        if end <= start:
            end = min(n_frames, start + 1)
        Y[pitch - MIN_MIDI, start:end] = 1.0
    return Y


def compute_cqt_features(audio_path):
    y, sr = librosa.load(audio_path, sr=SR, mono=True)
    y = librosa.util.normalize(y)
    # Keep CQT below Nyquist. If this errors, reduce N_BINS or FMIN in config.
    C = librosa.cqt(
        y, sr=sr, hop_length=HOP_LENGTH, fmin=FMIN,
        n_bins=N_BINS, bins_per_octave=BINS_PER_OCTAVE
    )
    X = librosa.amplitude_to_db(np.abs(C), ref=np.max).astype(np.float32)
    # Normalize to roughly 0-1 then standardize per clip.
    X = (X + 80.0) / 80.0
    X = np.clip(X, 0.0, 1.0)
    return X


def cache_record_features(record, audio_path, overwrite=False):
    out_path = record_cache_path(record['recording'])
    if out_path.exists() and not overwrite:
        return out_path
    X = compute_cqt_features(audio_path)
    Y = make_pitch_roll(record['notes'], X.shape[1])
    np.savez_compressed(out_path, X=X, Y=Y, recording=record['recording'], audio_path=str(audio_path))
    return out_path

all_paired = TRAIN_PAIRED + VAL_PAIRED + TEST_PAIRED
for record, audio_path in tqdm(all_paired, desc='Caching CQT/labels'):
    cache_record_features(record, audio_path, overwrite=False)

print('Cached features:', len(list(FEATURE_CACHE_DIR.glob('*_cqt_pitchroll.npz'))))


Caching CQT/labels:   0%|          | 0/360 [00:00<?, ?it/s]

Cached features: 360


## 5. Dataset windows and augmentation

Training uses overlapping windows so the model sees many examples from each 30-sec GuitarSet recording.


In [9]:
class GuitarSetPitchDataset(Dataset):
    def __init__(self, paired_records, train=False):
        self.train = train
        self.index = []
        self.window_frames = int(round(WINDOW_SECONDS * SR / HOP_LENGTH))
        self.hop_frames = int(round(WINDOW_HOP_SECONDS * SR / HOP_LENGTH))
        self.record_lookup = {r['recording']: r for r, _ in paired_records}
        for record, _ in paired_records:
            p = record_cache_path(record['recording'])
            data = np.load(p)
            n_frames = data['X'].shape[1]
            starts = list(range(0, max(1, n_frames - self.window_frames + 1), self.hop_frames))
            if not starts or starts[-1] + self.window_frames < n_frames:
                starts.append(max(0, n_frames - self.window_frames))
            for st in starts:
                self.index.append((record['recording'], st))

    def __len__(self):
        return len(self.index)

    def _augment(self, X, Y):
        if random.random() > AUG_PROB:
            return X, Y

        # CQT feature gain/noise masking.
        if random.random() < 0.5:
            X = X + np.random.normal(0, 0.015, size=X.shape).astype(np.float32)
        if random.random() < 0.5:
            # Frequency masking.
            f = random.randint(0, max(0, X.shape[0] - 12))
            w = random.randint(2, 10)
            X[f:f+w, :] *= random.uniform(0.2, 0.8)
        if random.random() < 0.5:
            # Time masking.
            t = random.randint(0, max(0, X.shape[1] - 16))
            w = random.randint(4, 16)
            X[:, t:t+w] *= random.uniform(0.2, 0.8)

        # Approx pitch shift by rolling CQT bins and pitch-roll labels.
        # 2 CQT bins per semitone because BINS_PER_OCTAVE=24.
        if random.random() < 0.35:
            shift = random.choice(PITCH_SHIFT_AUG_SEMITONES)
            if shift != 0:
                X = np.roll(X, shift * 2, axis=0)
                if shift > 0:
                    X[:shift*2, :] = 0
                    Y = np.roll(Y, shift, axis=0)
                    Y[:shift, :] = 0
                else:
                    X[shift*2:, :] = 0
                    Y = np.roll(Y, shift, axis=0)
                    Y[shift:, :] = 0
        X = np.clip(X, 0.0, 1.0)
        return X, Y

    def __getitem__(self, idx):
        recording, st = self.index[idx]
        p = record_cache_path(recording)
        data = np.load(p)
        X = data['X']
        Y = data['Y']
        end = st + self.window_frames
        if X.shape[1] < self.window_frames:
            pad = self.window_frames - X.shape[1]
            Xw = np.pad(X, ((0,0),(0,pad)), mode='constant')
            Yw = np.pad(Y, ((0,0),(0,pad)), mode='constant')
        else:
            Xw = X[:, st:end]
            Yw = Y[:, st:end]
        Xw = Xw.astype(np.float32).copy()
        Yw = Yw.astype(np.float32).copy()
        if self.train and USE_FEATURE_AUGMENTATION:
            Xw, Yw = self._augment(Xw, Yw)
        # Model input: [channels, freq, time]
        return torch.from_numpy(Xw[None, :, :]), torch.from_numpy(Yw.T)  # Y: [time, pitch]

train_ds = GuitarSetPitchDataset(TRAIN_PAIRED, train=True)
val_ds = GuitarSetPitchDataset(VAL_PAIRED, train=False)
test_ds = GuitarSetPitchDataset(TEST_PAIRED, train=False)
print('windows:', {'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds)})

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


windows: {'train': 1801, 'val': 388, 'test': 379}


## 6. Guitar-specific model

Lightweight CNN + BiGRU framewise pitch-roll model.

This is intentionally smaller/easier than the paper's conformer model so it can train in a Colab notebook. It is still guitar-specific and trained directly on GuitarSet labels.


In [10]:
class CNNBiGRUPitchModel(nn.Module):
    def __init__(self, n_pitches=N_PITCHES):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),  # reduce freq only
            nn.Dropout(0.10),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout(0.10),

            nn.Conv2d(64, 96, kernel_size=3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Dropout(0.10),
        )
        reduced_freq = N_BINS // 8
        self.proj = nn.Linear(96 * reduced_freq, 192)
        self.rnn = nn.GRU(
            input_size=192,
            hidden_size=160,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.15,
        )
        self.out = nn.Sequential(
            nn.Linear(320, 192),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(192, n_pitches),
        )

    def forward(self, x):
        # x: [B, 1, F, T]
        z = self.cnn(x)  # [B, C, F', T]
        z = z.permute(0, 3, 1, 2).contiguous()  # [B, T, C, F']
        z = z.view(z.shape[0], z.shape[1], -1)  # [B, T, C*F']
        z = self.proj(z)
        z, _ = self.rnn(z)
        logits = self.out(z)  # [B, T, pitches]
        return logits

model = CNNBiGRUPitchModel().to(DEVICE)
print(model)
print('parameters:', sum(p.numel() for p in model.parameters())/1e6, 'M')


CNNBiGRUPitchModel(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.1, inplace=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.1, inplace=False)
    (10): Conv2d(64, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (14): Dropout(p=0.1, inplace=False)
  )
  (proj): Li

## 7. Train

Uses BCE loss with positive weighting because active pitches are sparse.


In [11]:
def estimate_pos_weight(dataset, max_batches=100):
    pos = 0.0
    total = 0.0
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    for i, (_, y) in enumerate(loader):
        pos += y.sum().item()
        total += y.numel()
        if i + 1 >= max_batches:
            break
    neg = total - pos
    pw = neg / max(pos, 1.0)
    return min(max(pw, 1.0), 50.0)

POS_WEIGHT = estimate_pos_weight(train_ds)
print('Estimated positive weight:', POS_WEIGHT)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)


def frame_metrics_from_logits(logits, y, threshold=0.40):
    probs = torch.sigmoid(logits)
    pred = (probs >= threshold).float()
    tp = (pred * y).sum().item()
    fp = (pred * (1-y)).sum().item()
    fn = ((1-pred) * y).sum().item()
    precision = tp / max(tp + fp, 1e-9)
    recall = tp / max(tp + fn, 1e-9)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return precision, recall, f1


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    metrics = []
    for x, y in tqdm(loader, leave=False):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * x.shape[0]
        metrics.append(frame_metrics_from_logits(logits.detach(), y.detach(), threshold=0.40))
    avg_loss = total_loss / len(loader.dataset)
    avg_metrics = np.mean(metrics, axis=0).tolist()
    return avg_loss, avg_metrics

best_val_f1 = -1
history = []
ckpt_path = MODEL_DIR / 'best_cnn_bigru_pitch_model.pt'

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_m = run_epoch(train_loader, train=True)
    val_loss, val_m = run_epoch(val_loader, train=False)
    scheduler.step(val_m[2])
    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_precision': train_m[0],
        'train_recall': train_m[1],
        'train_f1': train_m[2],
        'val_precision': val_m[0],
        'val_recall': val_m[1],
        'val_f1': val_m[2],
        'seconds': time.time() - t0,
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(row)
    print(row)
    if val_m[2] > best_val_f1:
        best_val_f1 = val_m[2]
        torch.save({'model_state': model.state_dict(), 'config': {'MIN_MIDI': MIN_MIDI, 'MAX_MIDI': MAX_MIDI}}, ckpt_path)
        print('Saved best:', ckpt_path)

pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
print('Best val frame F1:', best_val_f1)


Estimated positive weight: 26.488512576882062


  0%|          | 0/151 [00:00<?, ?it/s]

  0%|          | 0/33 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.8637924155697169, 'val_loss': 0.5445924074072199, 'train_precision': 0.09310307272596924, 'train_recall': 0.9453952145639334, 'train_f1': 0.16782100365124344, 'val_precision': 0.1520634786562562, 'val_recall': 0.9669850814249042, 'val_f1': 0.2623461477212186, 'seconds': 33.13953232765198, 'lr': 0.0003}
Saved best: /content/drive/MyDrive/Capstone/outputs/guitarset_audio_to_midi_finetune_v1_fixed_cqt/models/best_cnn_bigru_pitch_model.pt


  0%|          | 0/151 [00:00<?, ?it/s]

...

  0%|          | 0/33 [00:00<?, ?it/s]

{'epoch': 19, 'train_loss': 0.10193253898044748, 'val_loss': 0.18386857695493503, 'train_precision': 0.5966986997904288, 'train_recall': 0.9907216217819577, 'train_f1': 0.7436963426119251, 'val_precision': 0.6737465336918254, 'val_recall': 0.9668746114273311, 'val_f1': 0.7913994476627468, 'seconds': 31.12149167060852, 'lr': 0.0003}


  0%|          | 0/151 [00:00<?, ?it/s]

  0%|          | 0/33 [00:00<?, ?it/s]

{'epoch': 20, 'train_loss': 0.10345598400210024, 'val_loss': 0.18801594145365597, 'train_precision': 0.5960522410664277, 'train_recall': 0.9902719667967468, 'train_f1': 0.7430343603904016, 'val_precision': 0.6887454557978079, 'val_recall': 0.9660196216367292, 'val_f1': 0.8016608829087976, 'seconds': 31.281243324279785, 'lr': 0.0003}
Best val frame F1: 0.8043614566786466


## 8. Full-record inference and note event extraction

This converts model frame probabilities back into note events:

```text
recording, onset, offset, pitch, amplitude
```

That is the same shape the rest of the pipeline expects from Basic Pitch.


In [12]:
def load_best_model():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.to(DEVICE)
    model.eval()
    return model

load_best_model()


def predict_record_pitch_probs(recording):
    p = record_cache_path(recording)
    data = np.load(p)
    X = data['X'].astype(np.float32)
    with torch.no_grad():
        x = torch.from_numpy(X[None, None, :, :]).to(DEVICE)
        logits = model(x)
        probs = torch.sigmoid(logits)[0].cpu().numpy().T  # [pitch, time]
    return probs, X.shape[1]


def pitchroll_to_note_events(probs, threshold=0.40, min_duration_sec=0.03, recording=None):
    events = []
    min_frames = max(1, int(round(min_duration_sec * SR / HOP_LENGTH)))
    for pi in range(probs.shape[0]):
        active = probs[pi] >= threshold
        if not np.any(active):
            continue
        padded = np.pad(active.astype(np.int8), (1, 1))
        changes = np.diff(padded)
        starts = np.where(changes == 1)[0]
        ends = np.where(changes == -1)[0]
        for st, en in zip(starts, ends):
            if en - st < min_frames:
                continue
            onset = st * HOP_LENGTH / SR
            offset = en * HOP_LENGTH / SR
            amp = float(probs[pi, st:en].max())
            events.append({
                'recording': recording,
                'onset': onset,
                'offset': offset,
                'duration': offset - onset,
                'pitch': int(MIN_MIDI + pi),
                'amplitude': amp,
            })
    events = sorted(events, key=lambda x: (x['onset'], x['pitch']))
    return events


def predict_record_events(recording, threshold=0.40, min_duration_sec=0.03):
    probs, _ = predict_record_pitch_probs(recording)
    return pitchroll_to_note_events(probs, threshold=threshold, min_duration_sec=min_duration_sec, recording=recording)


## 9. Note-level pitch/onset evaluation

This uses the same style of pitch + onset matching as your Basic Pitch experiments.


In [13]:
def match_pred_to_gt(pred_notes, gt_notes, onset_tolerance=ONSET_TOLERANCE_SECONDS):
    pred_notes = sorted(pred_notes, key=lambda x: (x['onset'], x['pitch']))
    gt_notes = sorted(gt_notes, key=lambda x: (x['onset'], x['pitch']))
    used_gt = set()
    matches = []
    false_pos = []

    for pi, pred in enumerate(pred_notes):
        best_j = None
        best_err = None
        for j, gt in enumerate(gt_notes):
            if j in used_gt:
                continue
            if int(pred['pitch']) != int(gt['pitch']):
                continue
            err = abs(float(pred['onset']) - float(gt['onset']))
            if err <= onset_tolerance and (best_err is None or err < best_err):
                best_j = j
                best_err = err
        if best_j is None:
            false_pos.append(pred)
        else:
            used_gt.add(best_j)
            gt = gt_notes[best_j]
            matches.append({
                'recording': pred.get('recording'),
                'pred_onset': pred['onset'],
                'gt_onset': gt['onset'],
                'onset_error': pred['onset'] - gt['onset'],
                'pitch': pred['pitch'],
                'pred_offset': pred.get('offset'),
                'gt_offset': gt.get('offset'),
                'amplitude': pred.get('amplitude'),
            })

    missed = [gt for j, gt in enumerate(gt_notes) if j not in used_gt]
    return matches, false_pos, missed


def evaluate_records(paired_records, threshold=0.40, min_duration_sec=0.03, split_name='eval'):
    rows = []
    all_pred, all_matches, all_fp, all_missed = [], [], [], []
    for record, _ in tqdm(paired_records, desc=f'Evaluating {split_name}'):
        pred = predict_record_events(record['recording'], threshold=threshold, min_duration_sec=min_duration_sec)
        matches, fp, missed = match_pred_to_gt(pred, record['notes'])
        tp = len(matches); n_pred = len(pred); n_gt = len(record['notes'])
        precision = tp / max(n_pred, 1)
        recall = tp / max(n_gt, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-9)
        rows.append({
            'split': split_name,
            'recording': record['recording'],
            'n_gt': n_gt,
            'n_pred': n_pred,
            'n_matched': tp,
            'n_false_positives': len(fp),
            'n_missed_gt_notes': len(missed),
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'mean_onset_abs_error': np.mean([abs(m['onset_error']) for m in matches]) if matches else np.nan,
        })
        for e in pred:
            all_pred.append(e)
        all_matches.extend(matches)
        all_fp.extend(fp)
        all_missed.extend(missed)

    metrics = pd.DataFrame(rows)
    total_gt = metrics['n_gt'].sum()
    total_pred = metrics['n_pred'].sum()
    total_match = metrics['n_matched'].sum()
    total_fp = metrics['n_false_positives'].sum()
    total_missed = metrics['n_missed_gt_notes'].sum()
    precision = total_match / max(total_pred, 1)
    recall = total_match / max(total_gt, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    summary = {
        'split': split_name,
        'threshold': threshold,
        'min_duration_sec': min_duration_sec,
        'n_records': len(metrics),
        'n_gt': int(total_gt),
        'n_pred': int(total_pred),
        'n_matched': int(total_match),
        'n_false_positives': int(total_fp),
        'n_missed_gt_notes': int(total_missed),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mean_onset_abs_error': metrics['mean_onset_abs_error'].mean(),
    }
    return summary, metrics, pd.DataFrame(all_pred), pd.DataFrame(all_matches), pd.DataFrame(all_fp), pd.DataFrame(all_missed)


## 10. Tune threshold on validation, then evaluate test

This is the defensible version:
- Choose threshold/min-duration using validation split
- Report final result on test split once


In [14]:
val_summaries = []
for th in THRESHOLD_GRID:
    for mdur in MIN_DURATION_GRID:
        summary, _, _, _, _, _ = evaluate_records(VAL_PAIRED, threshold=th, min_duration_sec=mdur, split_name='val')
        val_summaries.append(summary)

val_summary_df = pd.DataFrame(val_summaries).sort_values('f1', ascending=False)
val_summary_df.to_csv(OUTPUT_DIR / 'val_threshold_tuning.csv', index=False)
val_summary_df.head(10)


Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

...

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating val:   0%|          | 0/54 [00:00<?, ?it/s]

,split,threshold,min_duration_sec,n_records,n_gt,n_pred,n_matched,n_false_positives,n_missed_gt_notes,precision,recall,f1,mean_onset_abs_error
31,val,0.55,0.08,54,9154,6442,3832,2610,5322,0.594846,0.418615,0.491408,0.021758
30,val,0.55,0.05,54,9154,6745,3847,2898,5307,0.570348,0.420253,0.483930,0.021740
27,val,0.50,0.08,54,9154,6596,3774,2822,5380,0.572165,0.412279,0.479238,0.021997
29,val,0.55,0.03,54,9154,6972,3858,3114,5296,0.553356,0.421455,0.478482,0.021739
28,val,0.55,0.02,54,9154,6972,3858,3114,5296,0.553356,0.421455,0.478482,0.021739
26,val,0.50,0.05,54,9154,6931,3786,3145,5368,0.546242,0.413590,0.470749,0.021978
24,val,0.50,0.02,54,9154,7204,3794,3410,5360,0.526652,0.414464,0.463871,0.021957
25,val,0.50,0.03,54,9154,7204,3794,3410,5360,0.526652,0.414464,0.463871,0.021957
23,val,0.45,0.08,54,9154,6795,3693,3102,5461,0.543488,0.403430,0.463101,0.022137
22,val,0.45,0.05,54,9154,7151,3713,3438,5441,0.519228,0.405615,0.455443,0.022130


In [15]:
best = val_summary_df.iloc[0].to_dict()
BEST_THRESHOLD = float(best['threshold'])
BEST_MIN_DURATION = float(best['min_duration_sec'])
print('BEST_THRESHOLD:', BEST_THRESHOLD)
print('BEST_MIN_DURATION:', BEST_MIN_DURATION)
print('Best val F1:', best['f1'])

summary, by_record, pred_df, match_df, fp_df, missed_df = evaluate_records(
    TEST_PAIRED,
    threshold=BEST_THRESHOLD,
    min_duration_sec=BEST_MIN_DURATION,
    split_name='test'
)

print('TEST SUMMARY')
print(summary)

pd.DataFrame([summary]).to_csv(OUTPUT_DIR / 'test_summary_guitar_specific_model.csv', index=False)
by_record.to_csv(OUTPUT_DIR / 'test_metrics_by_record_guitar_specific_model.csv', index=False)
pred_df.to_csv(OUTPUT_DIR / 'test_predictions_guitar_specific_model.csv', index=False)
match_df.to_csv(OUTPUT_DIR / 'test_matches_guitar_specific_model.csv', index=False)
fp_df.to_csv(OUTPUT_DIR / 'test_false_positives_guitar_specific_model.csv', index=False)
missed_df.to_csv(OUTPUT_DIR / 'test_missed_gt_notes_guitar_specific_model.csv', index=False)

by_record.sort_values('f1').head(10)


BEST_THRESHOLD: 0.55
BEST_MIN_DURATION: 0.08
Best val F1: 0.491408053347012


Evaluating test:   0%|          | 0/54 [00:00<?, ?it/s]

TEST SUMMARY
{'split': 'test', 'threshold': 0.55, 'min_duration_sec': 0.08, 'n_records': 54, 'n_gt': 10207, 'n_pred': 6536, 'n_matched': 3909, 'n_false_positives': 2627, 'n_missed_gt_notes': 6298, 'precision': np.float64(0.5980722154222766), 'recall': np.float64(0.38297246987361616), 'f1': np.float64(0.46694140834975817), 'mean_onset_abs_error': np.float64(0.021859973788073268)}


,split,recording,n_gt,n_pred,n_matched,n_false_positives,n_missed_gt_notes,precision,recall,f1,mean_onset_abs_error
52,test,04_SS1-68-E_comp,733,45,14,31,719,0.311111,0.019100,0.035990,0.015798
51,test,00_Rock2-142-D_comp,547,127,34,93,513,0.267717,0.062157,0.100890,0.022762
50,test,00_Rock2-85-F_comp,816,166,56,110,760,0.337349,0.068627,0.114053,0.024588
44,test,05_Rock1-130-A_comp,312,75,27,48,285,0.360000,0.086538,0.139535,0.020392
37,test,03_Rock1-130-A_comp,212,88,27,61,185,0.306818,0.127358,0.180000,0.028838
42,test,04_Rock2-142-D_comp,353,124,46,78,307,0.370968,0.130312,0.192872,0.027243
40,test,02_Funk1-97-C_comp,189,167,47,120,142,0.281437,0.248677,0.264045,0.015384
45,test,05_Rock3-117-Bb_comp,398,154,74,80,324,0.480519,0.185930,0.268116,0.019213
28,test,03_SS1-68-E_comp,256,269,84,185,172,0.312268,0.328125,0.320000,0.024262
35,test,04_Rock1-130-A_comp,190,60,46,14,144,0.766667,0.242105,0.368000,0.022300


## 11. Optional: compare to your best Basic Pitch result

From the previous experiments, your best Basic Pitch-only result was about:

```text
F1 ≈ 0.787
Precision ≈ 0.861
Recall ≈ 0.724
```

Use this table to see whether the trained guitar-specific model is beating the post-processed Basic Pitch baseline.


In [16]:
comparison = pd.DataFrame([
    {
        'model': 'best_basic_pitch_v3_reference',
        'precision': 0.861,
        'recall': 0.724,
        'f1': 0.7867,
        'notes': 'Reference from prior V3 notebook, same 54-recording held-out split if split/pairing unchanged.'
    },
    {
        'model': 'guitar_specific_cnn_bigru',
        'precision': summary['precision'],
        'recall': summary['recall'],
        'f1': summary['f1'],
        'notes': 'Trained on GuitarSet train split; threshold tuned on val; evaluated on test.'
    }
])
comparison.to_csv(OUTPUT_DIR / 'comparison_vs_basic_pitch_reference.csv', index=False)
comparison


,model,precision,recall,f1,notes
0,best_basic_pitch_v3_reference,0.861000,0.724000,0.786700,"Reference from prior V3 notebook, same 54-reco..."
1,guitar_specific_cnn_bigru,0.598072,0.382972,0.466941,Trained on GuitarSet train split; threshold tu...


## 12. Export predictions for the existing tab pipeline

This writes predicted MIDI notes in a simple CSV shape that can be fed into the existing Viterbi/fretboard assignment step.


In [17]:
def export_record_predictions_for_pipeline(paired_records, split_name, threshold=BEST_THRESHOLD, min_duration_sec=BEST_MIN_DURATION):
    rows = []
    for record, audio_path in tqdm(paired_records, desc=f'Exporting {split_name}'):
        events = predict_record_events(record['recording'], threshold=threshold, min_duration_sec=min_duration_sec)
        for e in events:
            rows.append({
                'recording': record['recording'],
                'audio_path': str(audio_path),
                'start_time': e['onset'],
                'end_time': e['offset'],
                'duration': e['duration'],
                'pitch': e['pitch'],
                'amplitude': e['amplitude'],
                'source': 'guitar_specific_cnn_bigru',
            })
    df = pd.DataFrame(rows)
    out = OUTPUT_DIR / f'{split_name}_midi_notes_for_tab_pipeline.csv'
    df.to_csv(out, index=False)
    print('Wrote', out, 'rows:', len(df))
    return df

_ = export_record_predictions_for_pipeline(TEST_PAIRED, 'test')


Exporting test:   0%|          | 0/54 [00:00<?, ?it/s]

Wrote /content/drive/MyDrive/Capstone/outputs/guitarset_audio_to_midi_finetune_v1_fixed_cqt/test_midi_notes_for_tab_pipeline.csv rows: 6536


## 13. Next upgrade hooks

Once this baseline runs, the strongest next additions are:

1. Replace BiGRU with a small Conformer/Transformer encoder.
2. Add a note-level loss by pooling frame predictions around GuitarSet note onsets.
3. Add beat-informed quantization using GuitarSet beat/BPM annotations.
4. Add high-confidence Basic Pitch/Viterbi pseudo-labels for extra audio outside GuitarSet.
5. Train separate heads or thresholds for solo vs comp recordings.
